# 数据集样式
train:训练数据 里面只有一个正常样本good
test：测试图片  里面有很多文件夹既有good，也有hole等很多不同却显得文件夹
        在这里只做二分类，所以
        test/crack/*
        test/scratch/*
        ...
        都归属为Defect
ground_truth： 缺陷位置的标注。一般都是缺陷区域的mask（掩膜）。主要用于缺陷定位/分割/像素级评价，本次不用


Step 1：加载 train/good
Step 2：ResNet18 去掉最后分类层
Step 3：提取正常图片 feature
Step 4：计算正常 feature 中心
Step 5：读取 test
Step 6：计算 anomaly score
Step 7：确定 threshold
Step 8：输出 Good / Non-Good


In [1]:
import os.path
from xml.sax.handler import all_features

from torchvision import datasets, transforms
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    "dataset/leather/train",
    transform = transform
)
print("加载全数据集完成。。。")
print("总样本数：",len(train_dataset))
print("类别：",train_dataset.classes)
print("类别映射：",train_dataset.class_to_idx)

加载全数据集完成。。。
总样本数： 245
类别： ['good']
类别映射： {'good': 0}


In [2]:
from torchvision import models
model = models.resnet18(weights = models.ResNet18_Weights.DEFAULT)

# 去掉最后的全连接分类层  这里是直接获取前一层的512维特征，不要输出1000个分类的logits
model.fc = torch.nn.Identity()

model = model.to(device)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [3]:
image,label = train_dataset[0]
image = image.unsqueeze(0).to(device)

with torch.no_grad():
    feature = model(image)
print("feature shape:",feature.shape)

feature shape: torch.Size([1, 512])


In [4]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset= train_dataset,
    batch_size=32,
    shuffle=False,
)

all_features = []

with torch.no_grad():
    for images,labels in train_loader:
        images = images.to(device)
        features = model(images)
        all_features.append(features.cpu())
all_features = torch.cat(all_features,dim=0)
print("特征 feature shape:",all_features.shape)

特征 feature shape: torch.Size([245, 512])


In [5]:
normal_center = all_features.mean(dim=0)
print("正常特征中心 shape:", normal_center.shape)

正常特征中心 shape: torch.Size([512])


In [6]:
# 自定义dataset
from torch.utils.data import Dataset
from PIL import Image
class MVTecTestDataTest(Dataset):
    def __init__(self,root,transform=None):
        self.root = root
        self.transform = transform

        self.samples = []

        for defect_type in sorted(os.listdir(root)):
            defect_dir = os.path.join(root,defect_type)

            if not os.path.isdir(defect_dir):
                continue
            # 除了good都是1
            label = 0 if defect_type=='good' else 1

            for filename in sorted(os.listdir(defect_dir)):
                if filename.lower().endswith((".png",".jpg",".jpeg")):
                    image_path = os.path.join(defect_dir,filename)
                    self.samples.append((image_path,label,defect_type))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path,label,defect_type = self.samples[index]
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image,label,defect_type,image_path

In [7]:
test_dataset = MVTecTestDataTest(
    'dataset/leather/test',
    transform = transform
)
print("测试集数据：",len(test_dataset))
for i in range(5):
    image, label, defect_type, path = test_dataset[i]

    print(
        "label:", label,
        "type:", defect_type,
        "path:", path
    )

测试集数据： 124
label: 1 type: color path: dataset/leather/test\color\000.png
label: 1 type: color path: dataset/leather/test\color\001.png
label: 1 type: color path: dataset/leather/test\color\002.png
label: 1 type: color path: dataset/leather/test\color\003.png
label: 1 type: color path: dataset/leather/test\color\004.png


# 计算欧氏距离

In [8]:
import torch

test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
)

all_scores = []
all_labels = []
all_types = []

with torch.no_grad():
    for images,labels,defect_types,paths in test_dataloader:
        images = images.to(device)

        features = model(images)

        # 计算欧式距离
        distances = torch.norm(
            features.cpu() - normal_center,
            dim =1,
        )
        all_scores.extend(distances.tolist())
        all_labels.extend(labels.tolist())
        all_types.extend(defect_types)
print("测试图片数量：", len(all_scores))
print(all_scores[:20])

测试图片数量： 124
[9.226820945739746, 6.196648597717285, 4.552480220794678, 5.324605464935303, 3.325601100921631, 4.92338228225708, 4.515439033508301, 3.972716808319092, 8.363040924072266, 3.8927154541015625, 3.9263405799865723, 5.0479888916015625, 4.965970993041992, 3.618044376373291, 4.042840480804443, 6.991019248962402, 4.20278787612915, 5.849790573120117, 4.331046104431152, 5.527410507202148]


# 分别统计Good和Non-Good


In [9]:
import numpy as np
good_scores = [
    score for score,label in zip(all_scores,all_labels)
    if label==0
]
non_good_scores = [
    score for score,label in zip(all_scores,all_labels)
    if label==1
]
print("Good 数量：", len(good_scores))
print("Non-Good 数量：", len(non_good_scores))

print("\nGood:")
print("最小值：", np.min(good_scores))
print("最大值：", np.max(good_scores))
print("平均值：", np.mean(good_scores))

print("\nNon-Good:")
print("最小值：", np.min(non_good_scores))
print("最大值：", np.max(non_good_scores))
print("平均值：", np.mean(non_good_scores))

Good 数量： 32
Non-Good 数量： 92

Good:
最小值： 2.321220874786377
最大值： 4.575472831726074
平均值： 3.389375686645508

Non-Good:
最小值： 1.8829375505447388
最大值： 16.80919075012207
平均值： 6.138515346724054


# 观察结果
观察结果发现这两类分数有着明显重叠


In [10]:
results = []

for score,label,defect_type ,path in zip(all_scores,all_labels,all_types,[item[0] for item in test_dataset.samples]):
    results.append({
        "score":score,
        "label":label,
        "type":defect_type,
        "path":path
    })
results = sorted(results,key=lambda  x: x["score"])
print("分数最低的15个图片")
for item in results[:15]:
    print(item)

print("分数最高的15个图片")
for item in results[-15:]:
    print(item)

分数最低的15个图片
{'score': 1.8829375505447388, 'label': 1, 'type': 'glue', 'path': 'dataset/leather/test\\glue\\004.png'}
{'score': 2.221013307571411, 'label': 1, 'type': 'cut', 'path': 'dataset/leather/test\\cut\\014.png'}
{'score': 2.262502431869507, 'label': 1, 'type': 'cut', 'path': 'dataset/leather/test\\cut\\009.png'}
{'score': 2.321220874786377, 'label': 0, 'type': 'good', 'path': 'dataset/leather/test\\good\\004.png'}
{'score': 2.566851854324341, 'label': 0, 'type': 'good', 'path': 'dataset/leather/test\\good\\003.png'}
{'score': 2.587441921234131, 'label': 0, 'type': 'good', 'path': 'dataset/leather/test\\good\\021.png'}
{'score': 2.606860399246216, 'label': 0, 'type': 'good', 'path': 'dataset/leather/test\\good\\000.png'}
{'score': 2.6696207523345947, 'label': 0, 'type': 'good', 'path': 'dataset/leather/test\\good\\023.png'}
{'score': 2.73065447807312, 'label': 0, 'type': 'good', 'path': 'dataset/leather/test\\good\\010.png'}
{'score': 2.753549098968506, 'label': 0, 'type': 'good',

# 小步结论
目前看到有重叠区间，现在判断如何设置阈值
此时同样的又用到混淆矩阵

In [11]:
import numpy as np

scores = np.array(all_scores)
labels = np.array(all_labels)
print("threshold | TP | TN | FP | FN | Recall | Precision | F1")

for threshold in np.arange(1.0, 10.1, 0.5):

    # 分数 >= threshold → 判定为 Non-Good
    predictions = (scores >= threshold).astype(int)

    TP = np.sum((predictions == 1) & (labels == 1))
    TN = np.sum((predictions == 0) & (labels == 0))
    FP = np.sum((predictions == 1) & (labels == 0))
    FN = np.sum((predictions == 0) & (labels == 1))

    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    print(
        f"{threshold:8.1f} | "
        f"{TP:2d} | {TN:2d} | {FP:2d} | {FN:2d} | "
        f"{recall:.3f} | {precision:.3f} | {f1:.3f}"
    )

threshold | TP | TN | FP | FN | Recall | Precision | F1
     1.0 | 92 |  0 | 32 |  0 | 1.000 | 0.742 | 0.852
     1.5 | 92 |  0 | 32 |  0 | 1.000 | 0.742 | 0.852
     2.0 | 91 |  0 | 32 |  1 | 0.989 | 0.740 | 0.847
     2.5 | 89 |  1 | 31 |  3 | 0.967 | 0.742 | 0.840
     3.0 | 88 | 13 | 19 |  4 | 0.957 | 0.822 | 0.884
     3.5 | 85 | 20 | 12 |  7 | 0.924 | 0.876 | 0.899
     4.0 | 76 | 24 |  8 | 16 | 0.826 | 0.905 | 0.864
     4.5 | 62 | 31 |  1 | 30 | 0.674 | 0.984 | 0.800
     5.0 | 50 | 32 |  0 | 42 | 0.543 | 1.000 | 0.704
     5.5 | 43 | 32 |  0 | 49 | 0.467 | 1.000 | 0.637
     6.0 | 30 | 32 |  0 | 62 | 0.326 | 1.000 | 0.492
     6.5 | 25 | 32 |  0 | 67 | 0.272 | 1.000 | 0.427
     7.0 | 22 | 32 |  0 | 70 | 0.239 | 1.000 | 0.386
     7.5 | 21 | 32 |  0 | 71 | 0.228 | 1.000 | 0.372
     8.0 | 20 | 32 |  0 | 72 | 0.217 | 1.000 | 0.357
     8.5 | 17 | 32 |  0 | 75 | 0.185 | 1.000 | 0.312
     9.0 | 16 | 32 |  0 | 76 | 0.174 | 1.000 | 0.296
     9.5 | 13 | 32 |  0 | 79 | 0.141 | 1.00

TP = 坏品 → 成功识别为坏品  true Positive 正确识别成了目标类

TN = 好品 → 成功识别为好品  true Negative  正确识别成了非目标类

FP = 好品 → 错判成坏品      false positive 错误的识别成了目标类
FN = 坏品 → 错判成好品      true negative  正确的识别成了非目标类

In [12]:
import pandas as pd

for threshold in [2.5,3.0]:

    fn_results = [
        item for item in results
        if item["label"] == 1 and item["score"] < threshold
    ]

    print(f"\n===== threshold = {threshold} =====")
    print("漏检数量：", len(fn_results))

    # 统计每种缺陷漏检数量
    defect_count = {}

    for item in fn_results:
        defect_type = item["type"]
        defect_count[defect_type] = defect_count.get(defect_type, 0) + 1

    print("各缺陷类型漏检情况：")

    for defect_type, count in defect_count.items():
        print(f"{defect_type:>6}: {count}")


===== threshold = 2.5 =====
漏检数量： 3
各缺陷类型漏检情况：
  glue: 1
   cut: 2

===== threshold = 3.0 =====
漏检数量： 4
各缺陷类型漏检情况：
  glue: 1
   cut: 3


# 分析
肉眼还差错误图片，发现这些图片的异常区域都非常小。
因为本身用的是整个图的特征值，所以如果异常区域比较小，那么特征值是判断不出来的。
特征图输出的512维特征是对图片整体信息的高度压缩表示，所以可能最后会被稀释，而且我们的输入图片都会被resize。

下一步我们从global转换成patch

# 开始PatchCore
核心思想：从 CNN 中间层提取局部 Patch 特征，建立“正常 Patch 特征库”，再用最近邻距离判断测试 Patch 是否异常。
patchCore其实有限制的，基本上要固定位置、焦距。相对位置要大部分一致。如果训练都有立着的物体，识别用倾斜45°的就肯定不行。

In [13]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = torch.nn.Identity()
model = model.to(device)
model.eval()
def extract_feature_map(x):

    x = model.conv1(x)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpool(x)

    x = model.layer1(x)
    x = model.layer2(x)
    x = model.layer3(x)

    return x

images, labels = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():
    feature_map = extract_feature_map(images)

print(feature_map.shape)
patch_features = feature_map.permute(0, 2, 3, 1)

print("patch features:", patch_features.shape)

torch.Size([32, 256, 14, 14])
patch features: torch.Size([32, 14, 14, 256])


In [14]:
all_patch_features = []
with torch.no_grad():
    for imags, labels in train_loader:
        images = images.to(device)

        feature_map = extract_feature_map(images)
        # 在模型里默认的数据为  B C H W  这里改成 B H W C  此时通道放在最后面，表示每个空间位置都有一个256维特征
        patch_features = feature_map.permute(0,2,3,1)

        patch_features = patch_features.reshape(-1,256)

        all_patch_features.append(patch_features.cpu())
all_patch_features = torch.cat(all_patch_features,dim=0)# dim=0 纵向拼接（增加行数） dim=1 横向拼接累加（增加列数）
print(all_patch_features.shape)

torch.Size([50176, 256])


这里我们把目前256个图片×196个patch=50176个patch特征（其中每个patch里面都是256维特征）的特征库。这是一个全集！
现在我们已经拿到了50176个patch特征值了。现在就是判断测试图片和这些50176个不同patch做比较，判断距离

In [15]:
# 取测试集第一张图片
image, label, defect_type, path = test_dataset[0]

image = image.unsqueeze(0).to(device)

with torch.no_grad():
    feature_map = extract_feature_map(image)

    # [1, 256, 14, 14]
    patch_features = feature_map.permute(0, 2, 3, 1)

    # [1, 14, 14, 256]
    patch_features = patch_features.reshape(-1, 256)

print("test patch features:", patch_features.shape)
# 判断距离
distances = torch.cdist(
    patch_features.cpu(),
    all_patch_features
)

print(distances.shape)

test patch features: torch.Size([196, 256])
torch.Size([196, 50176])


现在目标图片经过模型以后会生成196个patch（里面也是256维），分别和50176个正常patch算距离。
同时取最小值。

为什么取最小值？因为我们要回答的问题是 这个测试patch像不像某个正常的patch
只要能找到一个非常像的patch就行。所以我们取最小值，这就是最近邻距离。

第二次对196个patch取max，这是想要知道“这张图片有没有某一个局部区域非常异常”
然后得到一个max值。这代表这张图片存在一个局部区域，在正常特征库里很难找到相似区域。

In [16]:
nearest_distances = distances.min(dim=1).values

print(nearest_distances.shape)
print(nearest_distances)

anomaly_score = nearest_distances.max()

print("anomaly score:", anomaly_score.item())

torch.Size([196])
tensor([1.4517, 1.4718, 1.3427, 1.2596, 1.3127, 1.1200, 1.0262, 1.1422, 1.1919,
        1.1345, 1.1976, 1.3268, 1.6762, 1.9263, 1.2964, 1.1862, 1.0248, 0.9757,
        1.0044, 0.9022, 0.9057, 0.8982, 0.9734, 1.0367, 1.0461, 1.1456, 1.3257,
        1.1799, 1.2528, 1.1094, 0.9117, 0.8385, 0.9170, 0.8650, 0.8025, 0.7740,
        0.8401, 0.8684, 1.0534, 1.0962, 1.2479, 1.0904, 1.2667, 1.0594, 1.0055,
        0.8475, 0.8260, 0.7257, 0.8156, 0.8403, 0.7982, 0.9273, 0.8769, 0.8591,
        0.9549, 1.0323, 1.4214, 1.1520, 0.9739, 0.8721, 0.9610, 0.8599, 0.8164,
        0.9324, 1.0246, 1.1107, 0.9648, 0.9373, 1.1093, 1.0510, 1.2452, 0.9355,
        1.0051, 0.9384, 0.9279, 0.8891, 0.9596, 1.0857, 1.3377, 1.5958, 1.4171,
        1.2239, 1.1488, 0.9777, 1.3573, 1.1018, 0.9248, 0.9189, 0.7902, 0.9419,
        1.0009, 1.3555, 2.0186, 2.5936, 1.8536, 1.3958, 1.1393, 1.0245, 1.3329,
        0.9946, 0.9067, 0.8268, 0.7494, 0.7414, 0.9283, 1.3570, 2.2072, 4.4695,
        3.1846, 1.5787

In [17]:
if anomaly_score >= threshold:
    print("Anomaly")
else:
    print("Good")

Anomaly


In [18]:
patch_results = []

with torch.no_grad():

    for image, label, defect_type, path in test_dataset:

        image = image.unsqueeze(0).to(device)

        # [1, 256, 14, 14]
        feature_map = extract_feature_map(image)

        # [1, 14, 14, 256]
        patch_features = feature_map.permute(0, 2, 3, 1)

        # [196, 256]
        patch_features = patch_features.reshape(-1, 256)

        # [196, 50176]
        distances = torch.cdist(
            patch_features.cpu(),
            all_patch_features
        )

        # 每个测试 Patch 到最近正常 Patch 的距离
        # [196]
        nearest_distances = distances.min(dim=1).values

        # 整张图片的 anomaly score
        anomaly_score = nearest_distances.max().item()

        patch_results.append({
            "score": anomaly_score,
            "label": label,
            "type": defect_type,
            "path": path
        })

In [19]:
from collections import defaultdict
import numpy as np

type_scores = defaultdict(list)

for item in patch_results:
    type_scores[item["type"]].append(item["score"])

for defect_type, scores in type_scores.items():

    print(
        defect_type,
        "min:", round(min(scores), 3),
        "max:", round(max(scores), 3),
        "mean:", round(np.mean(scores), 3)
    )

color min: 1.936 max: 4.828 mean: 3.371
cut min: 2.323 max: 5.319 mean: 3.77
fold min: 2.204 max: 4.35 mean: 3.077
glue min: 1.967 max: 8.482 mean: 5.911
good min: 1.404 max: 2.266 mean: 1.661
poke min: 3.124 max: 5.798 mean: 3.985


In [20]:
import numpy as np

scores = np.array([
    item["score"]
    for item in patch_results
])

labels = np.array([
    item["label"]
    for item in patch_results
])

print("threshold | TP | TN | FP | FN | Recall | Precision | F1")

for threshold in np.arange(1.5, 6.1, 0.2):

    predictions = (scores >= threshold).astype(int)

    TP = np.sum((predictions == 1) & (labels == 1))
    TN = np.sum((predictions == 0) & (labels == 0))
    FP = np.sum((predictions == 1) & (labels == 0))
    FN = np.sum((predictions == 0) & (labels == 1))

    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
    )

    print(
        f"{threshold:8.2f} | "
        f"{TP:2d} | "
        f"{TN:2d} | "
        f"{FP:2d} | "
        f"{FN:2d} | "
        f"{recall:.3f} | "
        f"{precision:.3f} | "
        f"{f1:.3f}"
    )

threshold | TP | TN | FP | FN | Recall | Precision | F1
    1.50 | 92 |  7 | 25 |  0 | 1.000 | 0.786 | 0.880
    1.70 | 92 | 20 | 12 |  0 | 1.000 | 0.885 | 0.939
    1.90 | 92 | 28 |  4 |  0 | 1.000 | 0.958 | 0.979
    2.10 | 90 | 30 |  2 |  2 | 0.978 | 0.978 | 0.978
    2.30 | 87 | 32 |  0 |  5 | 0.946 | 1.000 | 0.972
    2.50 | 84 | 32 |  0 |  8 | 0.913 | 1.000 | 0.955
    2.70 | 79 | 32 |  0 | 13 | 0.859 | 1.000 | 0.924
    2.90 | 75 | 32 |  0 | 17 | 0.815 | 1.000 | 0.898
    3.10 | 62 | 32 |  0 | 30 | 0.674 | 1.000 | 0.805
    3.30 | 57 | 32 |  0 | 35 | 0.620 | 1.000 | 0.765
    3.50 | 55 | 32 |  0 | 37 | 0.598 | 1.000 | 0.748
    3.70 | 47 | 32 |  0 | 45 | 0.511 | 1.000 | 0.676
    3.90 | 38 | 32 |  0 | 54 | 0.413 | 1.000 | 0.585
    4.10 | 31 | 32 |  0 | 61 | 0.337 | 1.000 | 0.504
    4.30 | 30 | 32 |  0 | 62 | 0.326 | 1.000 | 0.492
    4.50 | 25 | 32 |  0 | 67 | 0.272 | 1.000 | 0.427
    4.70 | 20 | 32 |  0 | 72 | 0.217 | 1.000 | 0.357
    4.90 | 18 | 32 |  0 | 74 | 0.196 | 1.00

现在调成2.3的阈值看看FN的数据，即错误认定为good的这些图片

In [21]:
global_threshold = 2.3

global_fn = [
    r for r in patch_results
    if r["label"] == 1 and r["score"] <= global_threshold
]

print("Global FN:", len(global_fn))

for r in global_fn:
    print(r["type"], r["score"], r["path"])

Global FN: 5
color 2.105377674102783 dataset/leather/test\color\004.png
color 1.9364523887634277 dataset/leather/test\color\009.png
fold 2.204068899154663 dataset/leather/test\fold\010.png
glue 1.9669452905654907 dataset/leather/test\glue\004.png
glue 2.1005918979644775 dataset/leather/test\glue\010.png


In [22]:
# 把FN的patch分数对应起来
patch_dict = {
    r["path"]: r["score"]
    for r in patch_results
}

for r in global_fn:

    patch_score = patch_dict[r["path"]]

    print(
        f"type={r['type']:<6} "
        f"global={r['score']:.3f} "
        f"patch={patch_score:.3f} "
        f"{r['path']}"
    )

type=color  global=2.105 patch=2.105 dataset/leather/test\color\004.png
type=color  global=1.936 patch=1.936 dataset/leather/test\color\009.png
type=fold   global=2.204 patch=2.204 dataset/leather/test\fold\010.png
type=glue   global=1.967 patch=1.967 dataset/leather/test\glue\004.png
type=glue   global=2.101 patch=2.101 dataset/leather/test\glue\010.png


这5个图片就是 其实是缺陷图片，但却被识别成good的图片。通过图片观察可以看到，有的是异常区域很小，有的是异常区域细长。

In [23]:
# 转换成 图片路径-分数的字典
patch_dict = {
    item["path"]: item["score"]
    for item in patch_results
}

print("Global vs Patch")

for item in results:
    # 遍历，根据图片路径找到对应的patch分数
    patch_score = patch_dict[item["path"]]

    print(
        f"{item['path']:<45} "
        f"Global={item['score']:.10f} "
        f"Patch={patch_score:.10f}"
    )

Global vs Patch
dataset/leather/test\glue\004.png             Global=1.8829375505 Patch=1.9669452906
dataset/leather/test\cut\014.png              Global=2.2210133076 Patch=2.3293664455
dataset/leather/test\cut\009.png              Global=2.2625024319 Patch=2.5712239742
dataset/leather/test\good\004.png             Global=2.3212208748 Patch=1.8143624067
dataset/leather/test\good\003.png             Global=2.5668518543 Patch=1.5841438770
dataset/leather/test\good\021.png             Global=2.5874419212 Patch=1.4463204145
dataset/leather/test\good\000.png             Global=2.6068603992 Patch=1.9387139082
dataset/leather/test\good\023.png             Global=2.6696207523 Patch=1.4671154022
dataset/leather/test\good\010.png             Global=2.7306544781 Patch=1.6170902252
dataset/leather/test\good\015.png             Global=2.7535490990 Patch=1.8398852348
dataset/leather/test\good\017.png             Global=2.7568981647 Patch=1.4681103230
dataset/leather/test\good\019.png             Glo

现在开始做异常热力图
这张图片“哪里最像异常”
我们最后计算的时候是拿到196（坐标为 14 ×14）个patch，

In [24]:
import torch.nn.functional as F

anomaly_map = nearest_distances.reshape(1, 1, 14, 14)
# 原始patch 只有14×14 ，这里需要扩展到224
anomaly_map = F.interpolate(
    anomaly_map,
    size=(224, 224),
    mode="bilinear",
    align_corners=False
)

anomaly_map = anomaly_map.squeeze()

print(anomaly_map.shape)
print(anomaly_map.max())
print(anomaly_map.min())



torch.Size([224, 224])
tensor(3.4559)
tensor(0.8278)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))

plt.imshow(
    anomaly_map.detach().cpu().numpy()
)

plt.colorbar(label="Anomaly score")
plt.axis("off")
plt.show()